<a href="https://colab.research.google.com/github/okiiisa/Network-Automation/blob/main/Network_Automation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#Okisa Bella 169982

!pip install --upgrade pip
!pip install requests netmiko napalm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 41.7 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 535.8/535.8 kB 12.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.4/642.4 kB 19.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 40.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 48.4 MB/s  0:00:00
  Created wheel for ncclient: filename=ncclient-0.7.0-py3-none-any.whl size=94121 sha256=60f5d1117dd226e7d1c3f223301742b4c40871da590e9ad04453228db35211ae
  Stored in directory: /root/.cache/pip/wheels/7a/4b

###Define the Device Credentials
We use a **Cisco DevNet Always-On Sandbox**. This is a real router that is always reachable over the internet

`ConnectHandler` acts as a universal adapter

###Roles of `ConnectHandler`
- Establishes SSH connection to network device
- Automatically handles the handshake (things like logging in, handles the `#` or `>` prompts.

**NB** The moment you specify a device type e.g., cisco_ios, `ConnectHandler` knows how to:
*   specifically navigate that specific OS
*   Enter configuration mode
*   Wait for the device to finish processing a command





In [3]:
from netmiko import ConnectHandler
import getpass

##Device details for Cisco DevNet Sandbox
device = {
    'device_type': 'cisco_xe',
    'host': 'devnetsandboxiosxec9k.cisco.com',
    'username': 'bella.okisa',
    #'password': 'U2_JXr5zG7_ffw', #Public since we aren't using getpass
    'password': getpass.getpass("Enter router password: "), #Using getpass for secure input
    'port': 22,
  }
print (f"Tageting: {device['host']}")

Enter router password: ··········
Tageting: devnetsandboxiosxec9k.cisco.com


## Retrieve Information (Show the Commands)
Goal:
*   Establish a connection
*   Pull the inetrface status
This is the "Read" part of the automation


In [4]:
try:

  #Establish SSH connection
  print("Connecting to the device ...")
  net_connect = ConnectHandler(**device)

  #Run a show command
  print("\n--- Interface Status ---")
  output = net_connect.send_command('show ip int brief')
  print(output)

  #Disconnect
  net_connect.disconnect()
  print("\nDisconnected Successfully.")

except Exception as e:
  print(f"Error connecting to the device: {e}")

Connecting to the device ...

--- Interface Status ---
Interface              IP-Address      OK? Method Status                Protocol
Vlan1                  unassigned      YES NVRAM  up                    up      
Vlan2                  10.0.0.1        YES manual up                    down    
Vlan3                  10.0.1.1        YES manual up                    down    
Vlan4                  10.0.2.1        YES manual up                    down    
Vlan5                  10.0.3.1        YES manual up                    down    
Vlan6                  10.0.4.1        YES manual up                    down    
Vlan7                  10.0.5.1        YES manual up                    down    
Vlan8                  10.0.6.1        YES manual up                    down    
Vlan9                  10.0.7.1        YES manual up                    down    
Vlan10                 10.0.8.1        YES manual up                    down    
Vlan11                 10.0.9.1        YES manual up  

# Pushing Configurations
Automations is particularly spectacular when you want to change settings
Next, we will create a new virtual interface called Loopback99

In [6]:
#Re-establish connections for the config change

#Redefine device with getpass for password to ensure correct authentication
device = {
    'device_type': 'cisco_xe',
    'host': 'devnetsandboxiosxec9k.cisco.com',
    'username': 'bella.okisa',
    'password': getpass.getpass("Enter router password: "),
    'port': 22,
}

net_connect = ConnectHandler(**device)

config_commands = [
    'interface Loopback99',
    'description Configured via Python Netmiko',
    'ip address 192.168.99.1 255.255.255.255'
]

print ("Sending configurations commands ...")
output = net_connect.send_config_set(config_commands)
print(output)

#Verify the change
verification = net_connect.send_command("Show IP interface | include loopback99")
print("\nVerification")
print(verification)


Enter router password: ··········
Sending configurations commands ...
configure terminal
Enter configuration commands, one per line.  End with CNTL/Z.
sw-l1-0(config)#interface Loopback99
sw-l1-0(config-if)#description Configured via Python Netmiko
sw-l1-0(config-if)#ip address 192.168.99.1 255.255.255.255
sw-l1-0(config-if)#end
sw-l1-0#

Verification



# Task: Apply a simple configuration such as a loopback to your switch

In [8]:
# Restablishing the connection
print("Reconnecting to the device...")
net_connect = ConnectHandler(**device)

config_commands_100 = [
    'interface Loopback100',
    'description Configured via Python Netmiko',
    'ip address 192.168.100.1 255.255.255.255',
]

try:
  print("Sending configurations commands...")
  output = net_connect.send_config_set(config_commands_100)
  output_2 = net_connect.send_command("Show ip interface brief")
  print(output)
  print(output_2)

  # Verify the change
  verification = net_connect.send_command("show ip interface brief | Include Loopback100")
  print("\nVerification")
  print(verification)

finally:
  net_connect.disconnect() #Always close the door when finished
  print("\nDisconnected Successfully.")

Reconnecting to the device...
Sending configurations commands...
configure terminal
Enter configuration commands, one per line.  End with CNTL/Z.
sandbox-zt(config)#interface Loopback100
sandbox-zt(config-if)#description Configured via Python Netmiko
sandbox-zt(config-if)#ip address 192.168.100.1 255.255.255.255
sandbox-zt(config-if)#end
sandbox-zt#
Interface              IP-Address      OK? Method Status                Protocol
Vlan1                  unassigned      YES NVRAM  up                    up      
Vlan2                  10.0.0.1        YES manual up                    down    
Vlan3                  10.0.1.1        YES manual up                    down    
Vlan4                  10.0.2.1        YES manual up                    down    
Vlan5                  10.0.3.1        YES manual up                    down    
Vlan6                  10.0.4.1        YES manual up                    down    
Vlan7                  10.0.5.1        YES manual up                    down    


# Automating VLAN Subinterfaces

We want to see how to create logical subinterfaces that tag traffic with specific VLAN IDs (802.1Q)


In [11]:
#Define VLAN-to-subinterface mapping
vlans_to_config = [
    {'id': 10, 'ip': '192.168.10.1', 'desc': 'SCES_VLAN'},
    {'id': 20, 'ip': '192.168.20.1', 'desc': 'SCES_VLAN'},
]

with ConnectHandler(**device) as net_connect:
  for vlan in vlans_to_config:
    print(f"Configuring VLAN {vlan['id']}...")
    config_commands = [
        f"vlan {vlan['id']}",
        f"name {vlan['desc']}",
        f"interface Vlan{vlan['id']}",
        f"description {vlan['desc']}",
        f"ip address {vlan['ip']} 255.255.255.252",
        "no shutdown"
    ]

    output = net_connect.send_config_set(config_commands)
    print(output)

    #Verififcations
    print("\nVerifying subinterfaces")
    verify = net_connect.send_command("Show ip interface brief | include GigabitEthernet2. ")
    print(verify)

Configuring VLAN 10...

sw-l1-0#configure terminal
Enter configuration commands, one per line.  End with CNTL/Z.
sw-l1-0(config)#vlan 10
sw-l1-0(config-vlan)#name SCES_VLAN
sw-l1-0(config-vlan)#interface Vlan10
sw-l1-0(config-if)#description SCES_VLAN
sw-l1-0(config-if)#ip address 192.168.10.1 255.255.255.252
sw-l1-0(config-if)#no shutdown
sw-l1-0(config-if)#end
sw-l1-0#

Verifying subinterfaces

Configuring VLAN 20...
configure terminal
Enter configuration commands, one per line.  End with CNTL/Z.
sw-l1-0(config)#vlan 20
sw-l1-0(config-vlan)#name SCES_VLAN
VLAN #20 and #10 have an identical name: SCES_VLAN
sw-l1-0(config-vlan)#interface Vlan20
sw-l1-0(config-if)#description SCES_VLAN
sw-l1-0(config-if)#ip address 192.168.20.1 255.255.255.252
sw-l1-0(config-if)#no shutdown
sw-l1-0(config-if)#end
sw-l1-0#

Verifying subinterfaces



#Multi-Device Configuration
We want to try automating multiple virtual instances at once
*   **Workflow**: Store  your device list in a devices.json or yaml
*   **Lab Task**: We want to write a Python script that loops through the file, connects to each device and pushed a standard "Golden COnfig" (e.g., setting up the same local user, banner, SSH timeout settings, etc)

#JSON
JavaScript Object Notation (JSON) is a standard format for storing and exchanging data using human-readable text.

We can create JSON scripts/ files for this lab inventory in either of these two methods:
1. Manually in a text editor such as Notepad++, TextEdit, or VS Code or
2. Generate using a Python Script as shown in the next cell wehere we create device inventory using JSON.





In [12]:
import json
import getpass

# Define our device using a list of dictionaries

device_list = [
    {
        'device_type': 'cisco_xe',
        'host': 'devnetsandboxiosxec9k.cisco.com',
        'username': 'bella.okisa',
        'password': getpass.getpass("Enter router password: "),
        'port': 22,
    },
    {
        "device_type": "cisco_xe",
        "host": "192.0.2.10",
        "username": "admin",
        "password": "C1sco12345"

    }
]

# Write the data to a file named 'devicesPY.json'
with open('devicesPY.json', 'w') as f:
  json.dump(device_list, f, indent=4)

print("devices.json has been created")

Enter router password: ··········
devices.json has been created


#Multi-Device Automation Script
The script that follows will load a JSON file, connect to each device in sequence, and pushes a set of configuration commands.

In [17]:
from netmiko import NetmikoTimeoutException, NetmikoAuthenticationException, ConnectHandler

#Load the device fleet from the JSON file

def load_devices(filepath):
  with open('/content/devicesPY.json', 'r') as f:
    return json.load(f)
#Define the configuration to be applied

config_commands_multiple = [
    'interface GigabitEthernet1/0/1',
    'no switchport',
    'no shutdown',
    'interface GigabitEthernet1/0/1.50',
    'encapsulation dot1q 50',
    'ip address 192.168.50.1 255.255.255.0',
    'description Automation_VLAN_50',
]

def run_automation():
  devices = load_devices('devicesPY.json')

  for device in devices:
    print (f"\n--- Connecting to {device['host']} ---")
    try:
      # We can establish connection using dictionary unpacking (**)
      with ConnectHandler(**device) as net_connect:
        print(f"Pushing configuration to {net_connect.find_prompt()}...")

        #Apply the configuration set
        output = net_connect.send_config_set(config_commands_multiple)
        print(output)

        # Save the running configuration to NVRAM

        net_connect.send_command('write memory')
        print("Configuration saved successfully.")

    except NetmikoTimeoutException:
      print(f"Error: Connection timed out for {device['host']}.")
    except NetMikoAuthenticationException:
      print(f"Error: Authentication failed for {device['host']}.")
    except Exception as e:
      print(f"An unexpected error occurred: (e)")

if __name__ == "__main__":
  run_automation()



--- Connecting to devnetsandboxiosxec9k.cisco.com ---
Pushing configuration to sw-l1-0#...

sw-l1-0#
sw-l1-0#
sw-l1-0#configure terminal
Enter configuration commands, one per line.  End with CNTL/Z.
sw-l1-0(config)#interface GigabitEthernet1/0/1
sw-l1-0(config-if)#no switchport
sw-l1-0(config-if)#no shutdown
sw-l1-0(config-if)#interface GigabitEthernet1/0/1.50
sw-l1-0(config-subif)#encapsulation dot1q 50
sw-l1-0(config-subif)#ip address 192.168.50.1 255.255.255.0
sw-l1-0(config-subif)#description Automation_VLAN_50
sw-l1-0(config-subif)#end
sw-l1-0#
Configuration saved successfully.

--- Connecting to 192.0.2.10 ---
Error: Connection timed out for 192.0.2.10.
